# ε-windowed bifurcation — the sparse, targeted fork term

The dense fork term (notebooks 01–02) penalises **all** co-hit segment pairs, which
is O(T³) dense, down-scales everything, and breaks the 1BQF. The fix (per the
Notion to-do): apply the penalty **only inside the ε acceptance window** — of all
the segments at a hit that are within ε of each other, you are punished for
choosing more than one. This couples only the **genuinely competing**
(near-collinear) continuations, so it is **sparse, targeted, and 1BQF-safe**.
$`A=(\gamma+\delta)I-C+\beta B_\varepsilon`$, with $`B_\varepsilon`$ = co-hit pairs of angle $`<\varepsilon`$.

In [1]:
import sys; sys.path.insert(0,"/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Bifurification")
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp, time
import matplotlib.pyplot as plt
import bif
plt.rcParams.update({"figure.dpi":110,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
OUT=Path(bif.__file__).resolve().parent/"outputs"; OUT.mkdir(parents=True,exist_ok=True)
TAU=bif.threshold(0,'off')

## 1. Sparsity restored
$`B_\varepsilon`$ is sparse (nnz ≈ O(n_seg) or less) — it does **not** break the
sparse-A invariant, unlike the dense fork $`B`$ (O(T³)).

In [2]:
rows=[]
for T in [20,50,100]:
    ev=bif.event(T); ham=bif.base_hamiltonian(ev); A0=ham.A.tocsr()
    Bd=bif.fork_graph(ham._segment_to_hit_ids)
    Be=bif.fork_graph_eps(ham._segment_to_hit_ids, ham._segment_vectors, bif.EPS)
    rows.append(dict(T=T,n_seg=ham.n_segments,nnz_C=A0.nnz,nnz_B_dense=int(Bd.nnz),nnz_B_eps=int(Be.nnz)))
inv=pd.DataFrame(rows); display(inv)
Tv=inv["T"].to_numpy()
fig,ax=plt.subplots(figsize=(7,4.6))
ax.loglog(Tv,inv.nnz_C,'o-',color="#2166ac",label="nnz(C) continuation ~ $T^2$")
ax.loglog(Tv,inv.nnz_B_dense,'s-',color="#d6604d",label="nnz(B) dense fork ~ $T^3$")
ax.loglog(Tv,np.maximum(inv.nnz_B_eps,1),'D-',color="#1b7837",label="nnz($B_ε$) ε-windowed (sparse)")
ax.set_xlabel("T (tracks)"); ax.set_ylabel("non-zeros"); ax.legend(fontsize=9)
ax.set_title("ε-windowing restores sparsity", fontweight="bold")
fig.tight_layout()
for e,dp in (("pdf",600),("png",300)): fig.savefig(OUT/f"eps_fork_sparsity.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved eps_fork_sparsity")

,T,n_seg,nnz_C,nnz_B_dense,nnz_B_eps
0,20,1600,1720,60800,0
1,50,10000,10312,980000,552
2,100,40000,40624,7920000,46


saved eps_fork_sparsity


## 2. Classical — targeted suppression, no collateral
At T=50 the base event has a few false positives (near-collinear bridges). The
ε-fork removes them (false-rate → 0) with negligible efficiency cost and **no
down-scaling** (median true/false unchanged) — contrast the dense fork, which
down-scales everything.

In [3]:
T=50; ev=bif.event(T); ham=bif.base_hamiltonian(ev); A0=ham.A.tocsr(); n=ham.n_segments
truth=bif.truth_mask(ev)
Be=bif.fork_graph_eps(ham._segment_to_hit_ids, ham._segment_vectors, bif.EPS)
Bd=bif.fork_graph(ham._segment_to_hit_ids)
BETAS=[0,0.25,0.5,1.0,2.0]
def sweep(B):
    out=[]
    for beta in BETAS:
        A=(A0+beta*B).tocsc(); b=bif.DELTA*np.ones(n)
        sol=bif.solve_classical(A,b); m=bif.metrics(sol,truth,TAU)
        out.append(dict(beta=beta,eff=m['segment_efficiency'],far=m['segment_false_rate'],
                        nFA=m['n_false_active'],auc=bif.auc(sol,truth),
                        medT=np.median(sol[truth]),medF=np.median(sol[~truth])))
    return pd.DataFrame(out)
Se=sweep(Be); Sd=sweep(Bd)
print("ε-windowed:"); display(Se.round(3))
fig,ax=plt.subplots(1,2,figsize=(13,4.6))
ax[0].plot(Se.beta,Se.eff,'o-',color="#1b7837",label="efficiency")
ax[0].plot(Se.beta,Se.far,'s-',color="#c51b7d",label="false-rate")
ax[0].plot(Se.beta,Se.medT,'^:',color="#1b7837",alpha=.6,label="median true")
ax[0].plot(Se.beta,Se.medF,'v:',color="#c51b7d",alpha=.6,label="median false")
ax[0].set_title("(a) ε-windowed: false-rate→0, no down-scaling",fontweight="bold")
ax[1].plot(Sd.beta,Sd.eff,'o-',color="#1b7837",label="efficiency")
ax[1].plot(Sd.beta,Sd.far,'s-',color="#c51b7d",label="false-rate")
ax[1].plot(Sd.beta,Sd.medT,'^:',color="#1b7837",alpha=.6,label="median true")
ax[1].plot(Sd.beta,Sd.medF,'v:',color="#c51b7d",alpha=.6,label="median false")
ax[1].set_title("(b) dense fork: everything down-scales (collateral)",fontweight="bold")
for a in ax: a.axhline(TAU,color="k",ls=":",lw=1); a.set_xlabel("β"); a.legend(fontsize=8); a.set_ylabel("metric / activation")
fig.tight_layout()
for e,dp in (("pdf",600),("png",300)): fig.savefig(OUT/f"eps_vs_dense_classical.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print(f"saved eps_vs_dense_classical (T={T}: base false-active={int(Se.nFA.iloc[0])} -> {int(Se.nFA.iloc[-1])})")

ε-windowed:


,beta,eff,far,nFA,auc,medT,medF
0,0.00,1.00,0.02,4,1.000,0.423,0.25
1,0.25,1.00,0.02,4,1.000,0.412,0.25
2,0.50,0.98,0.00,0,1.000,0.412,0.25
3,1.00,0.98,0.00,0,1.000,0.411,0.25
4,2.00,0.98,0.00,0,0.981,0.410,0.25


saved eps_vs_dense_classical (T=50: base false-active=4 -> 0)


## 3. Quantum — the 1BQF is preserved
A sparse fork keeps the false bulk on the notch and the circuit small, so the 1BQF
stays fast and its discrimination intact. (Real ε-forks appear at T ≥ 50, which is
beyond the statevector reach; here we add the *k* tightest co-hit fork edges to a
T = 10 event — a sparse fork of the same character — and compare to the dense fork
result from notebook 02: AUC_Q → 0.55, 86 s.)

In [4]:
from helpers import solve_quantum_statevector
T=10; ev=bif.event(T); ham=bif.base_hamiltonian(ev); A0=ham.A.tocsr(); n=ham.n_segments
truth=bif.truth_mask(ev); sh=np.asarray(ham._segment_to_hit_ids); vec=np.asarray(ham._segment_vectors)
pairs=[]
for col in (0,1):
    o=np.argsort(sh[:,col],kind='stable'); k=sh[o,col]; bnd=np.r_[0,np.where(np.diff(k))[0]+1,n]
    for a,c in zip(bnd[:-1],bnd[1:]):
        gp=o[a:c]
        if len(gp)<2: continue
        V=vec[gp]; cm=np.clip(V@V.T,-1,1); ii,jj=np.triu_indices(len(gp),1)
        for x,y,cc in zip(gp[ii],gp[jj],cm[ii,jj]): pairs.append((np.arccos(cc),int(x),int(y)))
pairs.sort()
def Bk(k):
    r=[p[1] for p in pairs[:k]]; c=[p[2] for p in pairs[:k]]
    return (sp.coo_matrix((np.ones(len(r)),(r,c)),shape=(n,n))+sp.coo_matrix((np.ones(len(r)),(c,r)),shape=(n,n))).tocsr()
qr=[]
for k in [0,10,30]:
    B=Bk(k) if k else sp.csr_matrix((n,n))
    A=(A0+1.0*B).tocsc(); b=bif.DELTA*np.ones(n)
    solC=bif.solve_classical(A,b)
    t0=time.time(); solQ,pa,ns=solve_quantum_statevector(A,b); dt=time.time()-t0
    solQ=np.abs(np.asarray(solQ[:n]))
    qr.append(dict(fork_edges=k,nnz=int(A.nnz),auc_C=bif.auc(solC,truth),auc_Q=bif.auc(solQ,truth),t_Q=dt))
Q=pd.DataFrame(qr); display(Q.round(3))
fig,ax=plt.subplots(figsize=(7.5,4.6))
ax.plot(Q.fork_edges,Q.auc_Q,'s-',color="#1b7837",lw=2,label="sparse ε-fork: AUC_Q (preserved)")
ax.scatter([Q.fork_edges.max()+5],[0.547],color="#d6604d",s=90,zorder=5,label="dense fork (nb02): AUC_Q≈0.55")
ax.axhline(0.5,color="k",ls=":",lw=1,label="random"); ax.set_ylim(0.4,1.05)
ax.set_xlabel("# sparse fork edges (T=10)"); ax.set_ylabel("quantum AUC")
ax.set_title("Sparse fork preserves the 1BQF; dense fork destroys it", fontweight="bold")
ax.legend(fontsize=9)
fig.tight_layout()
for e,dp in (("pdf",600),("png",300)): fig.savefig(OUT/f"eps_fork_quantum.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print(f"saved eps_fork_quantum (sparse fork t_Q={Q.t_Q.iloc[-1]:.1f}s vs dense 86s)")

,fork_edges,nnz,auc_C,auc_Q,t_Q
0,0,460,1.0,1.000,3.75
1,10,480,1.0,0.986,1.06
2,30,520,1.0,0.960,1.54


saved eps_fork_quantum (sparse fork t_Q=1.5s vs dense 86s)


## 4. Summary — the ε-windowed bifurcation is the right term

| property | dense fork (all co-hit) | **ε-windowed fork** |
|---|---|---|
| nnz | O(T³), breaks sparse-A | **O(n_seg), sparse** |
| classical effect | uniform down-scaling (eff & far collapse) | **targeted: false-rate → 0, no collateral** |
| quantum 1BQF | AUC → 0.55 (random), ~20× slower | **AUC ≈ 1, fast** |

Restricting the Denby fork penalty to the **acceptance window** ε turns it from a
blunt, quantum-breaking term into a **sparse, targeted** one: it suppresses exactly
the near-collinear false bridges (the genuine bifurcation ambiguities) while
leaving the rest of the spectrum — and the 1BQF — intact. This is the version to
carry forward for algorithm modification.